# Robustness and Safety

## Overview

Modern machine learning models can be highly accurate on clean data yet fragile under small, carefully chosen perturbations. This notebook studies **robustness** (resistance to adversarial and distributional perturbations) and **safety** (avoiding harmful failures such as poisoned models and silent out-of-distribution errors). We build attacks and defenses from scratch using only numpy so the mechanics are fully transparent.

---

## 1. Adversarial Examples

An adversarial example is an input $x_{adv}$ close to a clean input $x$ under some norm, yet classified differently by the model. Formally, for a classifier $f$ and a small budget $\epsilon$:

$$ x_{adv} = \arg\max_{\|x' - x\|_p \le \epsilon} J(\theta, x', y) $$

where $J$ is the loss, $\theta$ the model parameters, and $y$ the true label. The constraint set $\{x' : \|x' - x\|_p \le \epsilon\}$ is the perturbation ball; the $L_\infty$ ball is the most common choice.

---

## 2. FGSM (Fast Gradient Sign Method)

FGSM (Goodfellow et al., 2015) takes a single step in the direction that maximizes the loss:

$$ x_{adv} = x + \epsilon \cdot \operatorname{sign}\big(\nabla_x J(\theta, x, y)\big) $$

The sign operation guarantees the perturbation stays inside the $L_\infty$ ball of radius $\epsilon$. It is fast (one gradient evaluation) but relatively weak.

---

## 3. PGD (Projected Gradient Descent)

PGD (Madry et al., 2018) iterates FGSM-style steps and projects back onto the allowed ball after each step:

$$ x^{(t+1)} = \Pi_{B_\epsilon(x)}\Big( x^{(t)} + \alpha \cdot \operatorname{sign}\big(\nabla_x J(\theta, x^{(t)}, y)\big) \Big) $$

Here $\alpha$ is the step size, $\Pi_{B_\epsilon(x)}$ projects onto the $L_\infty$ ball $B_\epsilon(x)$ (an element-wise clip to $[x - \epsilon, x + \epsilon]$), and $x^{(0)}$ is usually $x$ plus random noise inside the ball. PGD is considered a strong first-order attack.

---

## 4. Carlini-Wagner (CW)

The CW attack solves an optimization that minimizes the perturbation size while forcing misclassification:

$$ \min_{\delta} \; \|\delta\|_2^2 + c \cdot g(x + \delta) $$

where $g$ is a margin-based objective that is non-positive only when the input is misclassified, and $c$ trades off perturbation size against attack success. CW produces very small perturbations and is a benchmark for evaluating defenses.

---

## 5. DeepFool

DeepFool estimates the minimal perturbation needed to cross the nearest decision boundary by linearizing the classifier around the current point and stepping toward the closest hyperplane, repeating until the label flips. It gives a tight estimate of the distance to the decision boundary.

---

## 6. AutoAttack

AutoAttack (Croce and Hein, 2020) is an ensemble of parameter-free attacks (two APGD variants, FAB, and the black-box Square Attack). Because it removes hyperparameter tuning and combines complementary attacks, it is the de facto standard for reporting robust accuracy and exposing gradient-masking defenses.

---

## 7. Defenses

- **Adversarial training**: train on adversarial examples generated on the fly (often PGD), solving a min-max objective $\min_\theta \mathbb{E}\big[\max_{\|\delta\|\le\epsilon} J(\theta, x+\delta, y)\big]$. This is the most reliable empirical defense.
- **Randomized smoothing**: convolve the classifier with Gaussian noise to build a smoothed classifier $g(x) = \arg\max_c \Pr_{\eta \sim \mathcal{N}(0,\sigma^2 I)}[f(x+\eta)=c]$, which comes with a provable $L_2$ certified radius.
- **Certified defenses**: methods (interval bound propagation, convex relaxations, randomized smoothing) that prove no perturbation within a radius can change the prediction, rather than only resisting known attacks.

---

## 8. Distribution Shift and OOD Detection

Real deployments face inputs drawn from a different distribution than training (covariate shift, label shift, concept drift). **Out-of-distribution (OOD) detection** flags such inputs so the model can abstain. Common scores include maximum softmax probability, energy scores, and Mahalanobis distance in feature space. Safe systems pair a robust classifier with an OOD gate.

---

## 9. Model Poisoning and Backdoor Attacks

**Data poisoning** corrupts the training set to degrade accuracy or implant behavior. **Backdoor (trojan) attacks** insert a hidden trigger pattern so that any input carrying the trigger is misclassified to an attacker-chosen target, while clean accuracy stays normal. Defenses include data sanitization, activation clustering, neuron pruning, and spectral signatures.

---


In [1]:
import numpy as np

rng = np.random.default_rng(0)

# Synthetic 2-class data: two Gaussian blobs in 2D
n = 400
X0 = rng.normal(loc=[-1.5, -1.5], scale=0.7, size=(n // 2, 2))
X1 = rng.normal(loc=[1.5, 1.5], scale=0.7, size=(n // 2, 2))
X = np.vstack([X0, X1])
y = np.hstack([np.zeros(n // 2), np.ones(n // 2)]).astype(np.float64)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Train logistic regression by plain gradient descent
w = np.zeros(2)
b = 0.0
lr = 0.1
for _ in range(500):
    p = sigmoid(X @ w + b)
    grad_w = X.T @ (p - y) / n
    grad_b = np.mean(p - y)
    w -= lr * grad_w
    b -= lr * grad_b

def predict(Xin):
    return (sigmoid(Xin @ w + b) >= 0.5).astype(np.float64)

def bce_loss(Xin, yin):
    p = np.clip(sigmoid(Xin @ w + b), 1e-9, 1 - 1e-9)
    return -np.mean(yin * np.log(p) + (1 - yin) * np.log(1 - p))

def input_grad(Xin, yin):
    # d loss / d x for binary cross-entropy with logistic model
    p = sigmoid(Xin @ w + b)
    # gradient of per-sample loss wrt input: (p - y) * w
    return (p - yin)[:, None] * w[None, :]

clean_acc = np.mean(predict(X) == y)
print('train accuracy:', round(float(clean_acc), 4))
print('loss:', round(float(bce_loss(X, y)), 4))


train accuracy: 1.0
loss: 0.0156


In [2]:
# FGSM from scratch: x_adv = x + epsilon * sign(grad_x J)
def fgsm(Xin, yin, epsilon):
    g = input_grad(Xin, yin)
    return Xin + epsilon * np.sign(g)

print('epsilon  adv_accuracy')
for eps in [0.0, 0.25, 0.5, 1.0, 2.0]:
    X_adv = fgsm(X, y, eps)
    adv_acc = np.mean(predict(X_adv) == y)
    print(f'{eps:5.2f}    {adv_acc:.4f}')


epsilon  adv_accuracy
 0.00    1.0000
 0.25    0.9975
 0.50    0.9725
 1.00    0.8400
 2.00    0.1700


In [3]:
# PGD: iterated FGSM steps with projection onto the L-infinity epsilon ball
def pgd(Xin, yin, epsilon, alpha, steps):
    lower = Xin - epsilon
    upper = Xin + epsilon
    # random start inside the ball
    X_adv = Xin + rng.uniform(-epsilon, epsilon, size=Xin.shape)
    X_adv = np.clip(X_adv, lower, upper)
    for _ in range(steps):
        g = input_grad(X_adv, yin)
        X_adv = X_adv + alpha * np.sign(g)
        # project back onto the epsilon L-infinity ball around the clean input
        X_adv = np.clip(X_adv, lower, upper)
    return X_adv

print('epsilon  pgd_adv_accuracy')
for eps in [0.25, 0.5, 1.0, 2.0]:
    X_adv = pgd(X, y, epsilon=eps, alpha=eps / 4.0, steps=20)
    adv_acc = np.mean(predict(X_adv) == y)
    print(f'{eps:5.2f}    {adv_acc:.4f}')


epsilon  pgd_adv_accuracy
 0.25    0.9975
 0.50    0.9725
 1.00    0.8400
 2.00    0.1700


In [4]:
# Adversarial training (conceptual): augment each batch with FGSM examples
# and take retraining steps. We reset weights and train on clean + adversarial.
w_at = np.zeros(2)
b_at = 0.0
train_eps = 1.0

def sig(z):
    return 1.0 / (1.0 + np.exp(-z))

def grad_x_at(Xin, yin):
    p = sig(Xin @ w_at + b_at)
    return (p - yin)[:, None] * w_at[None, :]

for step in range(500):
    # generate FGSM examples against the CURRENT robust model
    g = grad_x_at(X, y)
    X_fgsm = X + train_eps * np.sign(g)
    # augmented batch: clean + adversarial
    X_aug = np.vstack([X, X_fgsm])
    y_aug = np.hstack([y, y])
    p = sig(X_aug @ w_at + b_at)
    m = X_aug.shape[0]
    w_at -= lr * (X_aug.T @ (p - y_aug) / m)
    b_at -= lr * np.mean(p - y_aug)

def predict_at(Xin):
    return (sig(Xin @ w_at + b_at) >= 0.5).astype(np.float64)

# Evaluate robust model under FGSM at eps = train_eps
g = grad_x_at(X, y)
X_attack = X + train_eps * np.sign(g)
print('robust model clean acc:', round(float(np.mean(predict_at(X) == y)), 4))
print('robust model adv  acc:', round(float(np.mean(predict_at(X_attack) == y)), 4))
print('(compare to the non-robust adv accuracy from the FGSM cell)')


robust model clean acc: 1.0
robust model adv  acc: 0.8475
(compare to the non-robust adv accuracy from the FGSM cell)


In [5]:
# Production tooling: use a library instead of hand-rolled attacks.
# The blocks below are commented out because they need extra dependencies
# and a trained model. They show the typical API surface.

# --- Adversarial Robustness Toolbox (ART) ---
# from art.estimators.classification import PyTorchClassifier
# from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
#
# classifier = PyTorchClassifier(model=model, loss=loss_fn,
#                                input_shape=(1, 28, 28), nb_classes=10,
#                                optimizer=optimizer)
# attack = FastGradientMethod(estimator=classifier, eps=0.2)
# x_adv = attack.generate(x=x_test)
# pgd = ProjectedGradientDescent(estimator=classifier, eps=0.2,
#                               eps_step=0.05, max_iter=40)
# x_adv_pgd = pgd.generate(x=x_test)

# --- Foolbox ---
# import foolbox as fb
# fmodel = fb.PyTorchModel(model, bounds=(0, 1))
# attack = fb.attacks.LinfPGD()
# raw, clipped, is_adv = attack(fmodel, images, labels, epsilons=[0.03, 0.1])

print('See ART, Foolbox, and CleverHans for batteries-included attacks and defenses.')


See ART, Foolbox, and CleverHans for batteries-included attacks and defenses.


## Additional Learning Resources

### Papers

- Explaining and Harnessing Adversarial Examples (FGSM, Goodfellow et al.): https://arxiv.org/abs/1412.6572
- Towards Deep Learning Models Resistant to Adversarial Attacks (PGD, Madry et al.): https://arxiv.org/abs/1706.06083
- Certified Adversarial Robustness via Randomized Smoothing (Cohen et al.): https://arxiv.org/abs/1902.02918

### Books & Courses

- Adversarial Robustness - Theory and Practice: https://adversarial-ml-tutorial.org/

### Tools & Libraries

- Adversarial Robustness Toolbox (ART): https://github.com/Trusted-AI/adversarial-robustness-toolbox
- Foolbox: https://github.com/bethgelab/foolbox
- CleverHans: https://github.com/cleverhans-lab/cleverhans
